In [2]:

import duckdb
DB_PATH = "scrape.duckdb"
con = duckdb.connect(DB_PATH)

In [ ]:
con.execute('''
create or replace table dim_date as
select distinct scrape_date as date_id,
cast(scrape_date as date) as full_date,
year(cast(scrape_date as date)) as year,
month(cast(scrape_date as date)) as month,
day(cast(scrape_date as date)) as day,
dayofweek(cast(scrape_date as date)) as day_of_week,
cast(cast(date_trunc('week', cast(scrape_date as date)) as date) as varchar) as week_start
from clean_scrape
where scrape_date is not null
order by date_id;
''')

In [12]:
df1 = con.execute('''
select * from dim_date;
''').df()

In [13]:
df1.head()

,date_id,full_date,year,month,day,day_of_week,week_start
0,2026-06-01,2026-06-01,2026,6,1,1,2026-06-01
1,2026-06-02,2026-06-02,2026,6,2,2,2026-06-01
2,2026-06-03,2026-06-03,2026,6,3,3,2026-06-01
3,2026-06-04,2026-06-04,2026,6,4,4,2026-06-01
4,2026-06-05,2026-06-05,2026,6,5,5,2026-06-01


In [ ]:
con.execute('''
create or replace table dim_entity as
select entity_id, canonical_name, canonical_phone, categories as all_categories, n_raw_rows, first_seen
from entity_map
order by entity_id;
''')

In [15]:
df2 = con.execute('''
select * from dim_entity;
''').df()
df2.head()

,entity_id,canonical_name,canonical_phone,all_categories,n_raw_rows,first_seen
0,1,Cedar FinHub,4832838234,ERP,26,2026-06-01
1,2,Beacon Node BPO Services,8325162691,MSP Platform,22,2026-06-01
2,3,Ridgeway CarePlatform,7958302201,Medical,23,2026-06-01
3,4,Summit ForgeTech,6002894196,ERP,32,2026-06-01
4,5,RavenVerdict Tech,8176039312,Legal,35,2026-06-01


In [16]:
con.execute('''
create or replace table dim_location as
select row_number() over (order by country, city) as location_id,
city, country
from (select distinct city, country from clean_scrape where country is not null) t
order by country, city;
''')

In [18]:
df3 = con.execute('''
select * from dim_location;
''').df()
df3.head()

,location_id,city,country
0,1,-,AU
1,2,Melbourne,AU
2,3,Sydney,AU
3,4,NaN,AU
4,5,-,CA


In [39]:
con.execute('''
create or replace table dim_location as
select row_number() over (order by country, city) as location_id,
case when trim(city) in ('-', 'NaN', 'none', 'null', 'n/a', '')
    then null
    else city
end as city,
country
from (select distinct 
case when trim(city) in ('-', 'NaN', 'none', 'null', 'n/a', '')
    then null
    else city
end as city,
country from clean_scrape where country is not null) t
order by country, city;
''')

In [40]:
df4 = con.execute('''
select * from dim_location;
''').df()
df4.head(100)

,location_id,city,country
0,1,Melbourne,AU
1,2,Sydney,AU
2,3,NaN,AU
3,4,Calgary,CA
4,5,Toronto,CA
5,6,Vancouver,CA
6,7,NaN,CA
7,8,Berlin,DE
8,9,Munich,DE
9,10,NaN,DE


In [41]:
con.execute('''

create or replace table fact_scrape as
select row_number() over () as fact_id,
cs.scrape_date as date_id,
dc.category_id,
coalesce(em.entity_id, -1) as entity_id,
coalesce(dl.location_id, -1) as location_id,
cs.phone_clean,
cs.email_clean,
cs.website_clean,
cs.employee_count,
cs._source_file,
cs.scraped_at
from clean_scrape cs
left join dim_category dc ON cs.category = dc.category_name
left join entity_map em ON cs.phone_clean = em.canonical_phone
left join dim_location dl
on coalesce(cs.city, '__null__') = coalesce(dl.city, '__null__')
and cs.country = dl.country
where cs.scrape_date is not null;

''')

In [42]:
df5 = con.execute(''' select * from fact_scrape limit 10; ''').df()
df5.head()

,fact_id,date_id,category_id,entity_id,location_id,phone_clean,email_clean,website_clean,employee_count,_source_file,scraped_at
0,1,2026-06-01,1,1,34,4832838234,info@cedarfinhub.io,https://www.cedarfinhub.io,118,scrape_2026-06-01.csv,2026-06-01 02:45:54
1,2,2026-06-01,5,2,37,8325162691,sales@beaconnodebposervices.com,https://www.beaconnodebposervices.com,24,scrape_2026-06-01.csv,2026-06-01 04:36:26
2,3,2026-06-01,6,3,30,7958302201,sales@ridgewaycareplatform.com,https://ridgewaycareplatform.com,641,scrape_2026-06-01.csv,2026-06-01 02:49:23
3,4,2026-06-01,1,4,34,6002894196,sales@summitforgetech.net,summitforgetech.net,42,scrape_2026-06-01.csv,2026-06-01 02:58:41
4,5,2026-06-01,4,5,35,8176039312,hello@ravenverdicttech.com,https://ravenverdicttech.com,20,scrape_2026-06-01.csv,2026-06-01 02:08:09
